# 01 · Number Systems & Binary Arithmetic
**Course:** SLAO – Systèmes Logiques et Architecture des Ordinateurs · ISG Tunis
**Covers:** TD1 exercises 1–4, 6 and 11 (base conversions, natural integers, binary arithmetic)

---

## Objectives
1. Convert integers and fractions between bases 2, 8, 10 and 16, **showing every step**.
2. Encode natural integers on *N* bits and detect when a value does not fit.
3. Implement binary addition, subtraction, multiplication and division with a bit-level trace.
4. Validate each result against Python's built-in arithmetic (self-checking notebook).

## Contents
1. Setup
2. Positional notation and base → decimal
3. Decimal → base *b* (successive division, integer part)
4. Decimal fractions → base *b* (successive multiplication)
5. Grouping method: binary ↔ octal ↔ hexadecimal
6. Natural integers on *N* bits
7. Binary arithmetic (add, subtract, multiply, divide)
8. TD1 solutions, checked automatically

## 1 · Setup

In [1]:
from __future__ import annotations
from fractions import Fraction
import pandas as pd

DIGITS = "0123456789ABCDEF"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## 2 · Positional notation and base → decimal

A number written $d_{n-1}\dots d_1 d_0 , d_{-1}d_{-2}\dots$ in base $b$ has the value

$$\sum_{i} d_i \cdot b^{\,i}$$

The function below returns the value **and** a table of each digit's weight, which is exactly the
hand-written expansion used in the TD (`1×2³ + 0×2² + …`).

In [2]:
def from_base(text: str, base: int) -> tuple[Fraction, pd.DataFrame]:
    """Convert a string such as '1010.101' (given in `base`) to an exact value.

    Returns the value as a Fraction (no rounding error) and the weight table.
    """
    text = text.strip().upper()
    int_part, _, frac_part = text.partition(".")
    rows, total = [], Fraction(0)

    for pos, ch in enumerate(reversed(int_part)):            # weights b^0, b^1, ...
        d = DIGITS.index(ch)
        assert d < base, f"digit {ch!r} is not valid in base {base}"
        rows.append((ch, pos, Fraction(base) ** pos, d * Fraction(base) ** pos))
        total += d * Fraction(base) ** pos

    for pos, ch in enumerate(frac_part, start=1):            # weights b^-1, b^-2, ...
        d = DIGITS.index(ch)
        assert d < base, f"digit {ch!r} is not valid in base {base}"
        rows.append((ch, -pos, Fraction(base) ** -pos, d * Fraction(base) ** -pos))
        total += d * Fraction(base) ** -pos

    table = pd.DataFrame(rows, columns=["digit", "exponent", "weight", "contribution"])
    return total, table.iloc[::-1].reset_index(drop=True)      # most significant digit first


value, table = from_base("1010.101", 2)
print("1010.101 (2) =", float(value), "(10)")
table.assign(weight=table.weight.astype(str), contribution=table.contribution.astype(str))

1010.101 (2) = 10.625 (10)


,digit,exponent,weight,contribution
0,1,-3,1/8,1/8
1,0,-2,1/4,0
2,1,-1,1/2,1/2
3,1,3,8,8
4,0,2,4,0
5,1,1,2,2
6,0,0,1,0


## 3 · Decimal → base *b* (integer part)

**Method – successive division:** divide by $b$, keep the remainder, repeat on the quotient until it is 0.
The remainders read **from last to first** give the digits.

In [3]:
def int_to_base(n: int, base: int) -> tuple[str, pd.DataFrame]:
    """Successive-division algorithm. Returns the digit string and the step table."""
    assert n >= 0 and 2 <= base <= 16
    if n == 0:
        return "0", pd.DataFrame([(0, 0, 0)], columns=["dividend", "quotient", "remainder"])
    steps, q = [], n
    while q > 0:
        steps.append((q, q // base, q % base))
        q //= base
    digits = "".join(DIGITS[r] for _, _, r in reversed(steps))
    return digits, pd.DataFrame(steps, columns=["dividend", "quotient", "remainder"])


bits, steps = int_to_base(85, 2)
print(f"85 (10) = {bits} (2)")
steps

85 (10) = 1010101 (2)


,dividend,quotient,remainder
0,85,42,1
1,42,21,0
2,21,10,1
3,10,5,0
4,5,2,1
5,2,1,0
6,1,0,1


## 4 · Decimal fractions → base *b*

**Method – successive multiplication:** multiply the fractional part by $b$; the integer part of the
product is the next digit; keep the new fractional part. Stop when it reaches 0 **or** when the requested
precision is reached (many decimals, e.g. 0.347, never terminate in binary).

In [4]:
def frac_to_base(x: Fraction | float | str, base: int, max_digits: int) -> tuple[str, pd.DataFrame, bool]:
    """Fractional part -> digits. Returns (digits, step table, is_exact)."""
    f = Fraction(str(x)) if not isinstance(x, Fraction) else x
    assert 0 <= f < 1
    steps, out = [], []
    for _ in range(max_digits):
        if f == 0:
            break
        prod = f * base
        d = int(prod)                       # integer part = next digit
        steps.append((str(f), str(prod), DIGITS[d]))
        out.append(DIGITS[d])
        f = prod - d
    return "".join(out) or "0", pd.DataFrame(steps, columns=["fraction", "x base", "digit"]), f == 0


def decimal_to_base(x: str | float, base: int, frac_digits: int = 8) -> str:
    """Full conversion of a decimal number (integer + fractional parts)."""
    d = Fraction(str(x))
    ip = int(d)
    fp = d - ip
    s = int_to_base(ip, base)[0]
    if fp:
        s += "." + frac_to_base(fp, base, frac_digits)[0]
    return s


digits, steps, exact = frac_to_base("0.347", 2, max_digits=4)
print("0.347 ->", digits, "(truncated at 4 digits, exact:", exact, ")")
steps

0.347 -> 0101 (truncated at 4 digits, exact: False )


,fraction,x base,digit
0,347/1000,347/500,0
1,347/500,347/250,1
2,97/250,97/125,0
3,97/125,194/125,1


## 5 · Grouping method: binary ↔ octal ↔ hexadecimal

Since $8 = 2^3$ and $16 = 2^4$, convert by grouping bits **3 by 3 (octal)** or **4 by 4 (hexadecimal)**:
integer part from right to left, fractional part from left to right, padding with zeros.

In [5]:
def regroup(bin_text: str, group: int) -> str:
    """Binary string -> octal (group=3) or hexadecimal (group=4)."""
    ip, _, fp = bin_text.partition(".")
    ip = ip.zfill(-(-len(ip) // group) * group)                       # pad left
    fp = fp.ljust(-(-len(fp) // group) * group, "0") if fp else ""    # pad right
    conv = lambda s: "".join(DIGITS[int(s[i:i + group], 2)] for i in range(0, len(s), group))
    out = conv(ip).lstrip("0") or "0"
    return out + ("." + conv(fp) if fp else "")


def expand(text: str, group: int) -> str:
    """Octal / hexadecimal string -> binary (each digit becomes `group` bits)."""
    ip, _, fp = text.upper().partition(".")
    conv = lambda s: "".join(format(DIGITS.index(c), f"0{group}b") for c in s)
    out = conv(ip).lstrip("0") or "0"
    return out + ("." + conv(fp) if fp else "")


rows = []
for b in ["10111", "111010110", "101110101100", "101.101111"]:
    rows.append({"binary": b, "octal": regroup(b, 3), "hexadecimal": regroup(b, 4)})
pd.DataFrame(rows)

,binary,octal,hexadecimal
0,10111,27,17
1,111010110,726,1D6
2,101110101100,5654,BAC
3,101.101111,5.57,5.BC


## 6 · Natural integers on *N* bits

On $N$ bits, the representable naturals are $[\,0,\;2^N - 1\,]$.
`encode_unsigned` raises an error when a value does not fit, which is exactly the question
"*code on 8 bits **if possible**, then on 16 bits*".

In [6]:
def unsigned_range(n_bits: int) -> tuple[int, int]:
    return 0, 2 ** n_bits - 1


def encode_unsigned(value: int, n_bits: int) -> str:
    lo, hi = unsigned_range(n_bits)
    if not lo <= value <= hi:
        raise OverflowError(f"{value} does not fit on {n_bits} bits (range {lo}..{hi})")
    return format(value, f"0{n_bits}b")


print("8 bits :", unsigned_range(8))
print("16 bits:", unsigned_range(16))

rows = []
for v in (128, 136, 1024, 32769):
    row = {"value": v}
    for n in (8, 16):
        try:
            row[f"{n} bits"] = encode_unsigned(v, n)
        except OverflowError:
            row[f"{n} bits"] = "does not fit"
    rows.append(row)
pd.DataFrame(rows)

8 bits : (0, 255)
16 bits: (0, 65535)


,value,8 bits,16 bits
0,128,10000000,0000000010000000
1,136,10001000,0000000010001000
2,1024,does not fit,0000010000000000
3,32769,does not fit,1000000000000001


## 7 · Binary arithmetic

Operations are implemented on **bit strings** (not on Python ints) so the notebook shows how the
hardware-level algorithm works. Python's integers are only used afterwards, as an independent check.

| Operation | Rule recalled in the TD |
|---|---|
| Addition | `0+0=0`, `0+1=1`, `1+1=0 carry 1`, `1+1+1=1 carry 1` |
| Subtraction | `0-0=0`, `1-0=1`, `1-1=0`, `0-1=1 borrow 1` |
| Multiplication | shift-and-add of partial products |
| Division | long division by repeated compare/subtract |

In [7]:
def _pad(a: str, b: str) -> tuple[str, str]:
    n = max(len(a), len(b))
    return a.zfill(n), b.zfill(n)


def bin_add(a: str, b: str) -> tuple[str, pd.DataFrame]:
    """Column addition. The result is one bit longer if a final carry is produced."""
    a, b = _pad(a, b)
    carry, out, rows = 0, [], []
    for i in range(len(a) - 1, -1, -1):
        s = int(a[i]) + int(b[i]) + carry
        rows.append((len(a) - 1 - i, a[i], b[i], carry, s % 2, s // 2))
        out.append(str(s % 2))
        carry = s // 2
    if carry:
        out.append("1")
    trace = pd.DataFrame(rows, columns=["bit", "a", "b", "carry_in", "sum", "carry_out"])
    return "".join(reversed(out)), trace


def bin_sub(a: str, b: str) -> tuple[str, pd.DataFrame]:
    """Column subtraction a - b, with a >= b (unsigned)."""
    a, b = _pad(a, b)
    assert int(a, 2) >= int(b, 2), "a < b: use two's complement (notebook 02)"
    borrow, out, rows = 0, [], []
    for i in range(len(a) - 1, -1, -1):
        d = int(a[i]) - int(b[i]) - borrow
        new_borrow = 1 if d < 0 else 0
        d += 2 * new_borrow
        rows.append((len(a) - 1 - i, a[i], b[i], borrow, d, new_borrow))
        out.append(str(d))
        borrow = new_borrow
    trace = pd.DataFrame(rows, columns=["bit", "a", "b", "borrow_in", "diff", "borrow_out"])
    return "".join(reversed(out)), trace


def bin_mul(a: str, b: str) -> tuple[str, list[str]]:
    """Shift-and-add multiplication. Returns the product and the partial products."""
    partials, acc = [], "0"
    for shift, bit in enumerate(reversed(b)):
        pp = (a if bit == "1" else "0" * len(a)) + "0" * shift
        partials.append(pp)
        acc, _ = bin_add(acc, pp)
    return acc.lstrip("0") or "0", partials


def bin_div(a: str, b: str) -> tuple[str, str, pd.DataFrame]:
    """Long division. Returns (quotient, remainder, step table)."""
    assert int(b, 2) != 0, "division by zero"
    rem, quo, rows = "0", [], []
    for bit in a:
        rem = (rem + bit).lstrip("0") or "0"           # bring down the next bit
        if int(rem, 2) >= int(b, 2):
            rem_before, rem = rem, bin_sub(rem, b)[0].lstrip("0") or "0"
            quo.append("1")
            rows.append((bit, rem_before, "subtract", rem))
        else:
            quo.append("0")
            rows.append((bit, rem, "-", rem))
    return "".join(quo).lstrip("0") or "0", rem, pd.DataFrame(rows, columns=["bit_down", "partial", "action", "remainder"])

### 7.1 Worked example – multiplication `1001 × 110`

In [8]:
product, partials = bin_mul("1001", "110")
w = len(product)
for pp in partials:
    print(pp.rjust(w))
print("-" * w)
print(product.rjust(w), f"  = {int('1001', 2)} x {int('110', 2)} = {int(product, 2)}")

  0000
 10010
100100
------
110110   = 9 x 6 = 54


### 7.2 Worked example – division `10010000111 ÷ 1011`

In [9]:
q, r, trace = bin_div("10010000111", "1011")
print(f"quotient = {q}   remainder = {r}")
trace

quotient = 1101001   remainder = 100


,bit_down,partial,action,remainder
0,1,1,-,1
1,0,10,-,10
2,0,100,-,100
3,1,1001,-,1001
4,0,10010,subtract,111
5,0,1110,subtract,11
6,0,110,-,110
7,0,1100,subtract,1
8,1,11,-,11
9,1,111,-,111


## 8 · TD1 solutions (self-checking)

Every cell below **asserts** the result against the published correction. If a cell runs without an
`AssertionError`, the notebook agrees with the correction.

### Exercise 1 – decimal → binary

In [10]:
expected = {85: "1010101", 100: "1100100", 145: "10010001"}
for n, exp in expected.items():
    got, _ = int_to_base(n, 2)
    assert got == exp, (n, got, exp)
    print(f"{n:>4} (10) = {got} (2)   OK")

got = decimal_to_base("78.347", 2, frac_digits=4)
assert got == "1001110.0101"
print("78.347 (10) =", got, "(2)  [4 digits after the point]   OK")

  85 (10) = 1010101 (2)   OK
 100 (10) = 1100100 (2)   OK
 145 (10) = 10010001 (2)   OK
78.347 (10) = 1001110.0101 (2)  [4 digits after the point]   OK


> **Note.** In the handwritten solution sheet, `85` is written `10101010`. That is 8 digits for a 7-bit result:
> the correct value is **`1010101`** (64 + 16 + 4 + 1), which is what the official correction and the code give.

### Exercise 2 – binary → decimal

In [11]:
for text, exp in [("1011", "11"), ("1101101", "109"), ("1010.101", "10.625")]:
    val, _ = from_base(text, 2)
    assert val == Fraction(exp), (text, val)
    print(f"{text:>9} (2) = {float(val)} (10)   OK")

     1011 (2) = 11.0 (10)   OK
  1101101 (2) = 109.0 (10)   OK
 1010.101 (2) = 10.625 (10)   OK


### Exercise 3 – binary → octal and hexadecimal

In [12]:
expected = {
    "10111":        ("27",   "17"),
    "111010110":    ("726",  "1D6"),
    "101110101100": ("5654", "BAC"),
    "101.101111":   ("5.57", "5.BC"),
}
for b, (o, h) in expected.items():
    assert regroup(b, 3) == o and regroup(b, 4) == h
    assert expand(o, 3).lstrip("0") == b.lstrip("0") or "." in b     # reverse check on integers
print("All octal / hexadecimal conversions match the correction.")
pd.DataFrame([{"binary": b, "octal": o, "hex": h} for b, (o, h) in expected.items()])

All octal / hexadecimal conversions match the correction.


,binary,octal,hex
0,10111,27,17
1,111010110,726,1D6
2,101110101100,5654,BAC
3,101.101111,5.57,5.BC


### Exercise 4 – natural integers on 8 and 16 bits

In [13]:
assert unsigned_range(8) == (0, 255) and unsigned_range(16) == (0, 65535)
assert encode_unsigned(128, 8) == "10000000"
assert encode_unsigned(136, 8) == "10001000"
assert encode_unsigned(1024, 16) == "0000010000000000"
assert encode_unsigned(32769, 16) == "1000000000000001"
try:
    encode_unsigned(1024, 8)
except OverflowError as e:
    print("Expected failure ->", e)

Expected failure -> 1024 does not fit on 8 bits (range 0..255)


### Exercise 6 – the four binary operations

In [14]:
# Addition: 010001101 + 101111111  (the result needs one more bit -> capacity overflow on 9 bits)
s, _ = bin_add("010001101", "101111111")
assert s == "1000001100" and len(s) == 10
print("addition       :", s, "<- 10 bits for 9-bit operands: capacity overflow")

# Subtraction: 10100110 - 00111100
d, _ = bin_sub("10100110", "00111100")
assert d == "01101010"
print("subtraction    :", d)

# Multiplication: 1001 x 110
p, _ = bin_mul("1001", "110")
assert p == "110110"
print("multiplication :", p)

# Division: 10010000111 / 1011
q, r, _ = bin_div("10010000111", "1011")
assert (q, r) == ("1101001", "100")
print("division       : quotient", q, " remainder", r)

# Independent check with Python integers
assert int(s, 2) == 0b010001101 + 0b101111111
assert int(d, 2) == 0b10100110 - 0b00111100
assert int(p, 2) == 0b1001 * 0b110
assert (int(q, 2), int(r, 2)) == divmod(0b10010000111, 0b1011)
print("\nPython integer arithmetic agrees with every bit-level result.")

addition       : 1000001100 <- 10 bits for 9-bit operands: capacity overflow
subtraction    : 01101010
multiplication : 110110
division       : quotient 1101001  remainder 100

Python integer arithmetic agrees with every bit-level result.


### Exercise 11 (first part) – hexadecimal with fraction → decimal and binary

In [15]:
value, _ = from_base("5B7.A34", 16)
print("5B7.A34 (16) =", float(value), "(10)")
assert value == Fraction("1463.6376953125")

binary = expand("5B7.A34", 4)
print("5B7.A34 (16) =", binary, "(2)")
assert binary.rstrip("0") == "10110110111.101000110100".rstrip("0")

5B7.A34 (16) = 1463.6376953125 (10)
5B7.A34 (16) = 10110110111.101000110100 (2)


## Summary

| Task | Algorithm | Function |
|---|---|---|
| base *b* → decimal | sum of digit × weight | `from_base` |
| decimal integer → base *b* | successive division | `int_to_base` |
| decimal fraction → base *b* | successive multiplication | `frac_to_base` |
| binary ↔ octal / hex | group 3 / 4 bits | `regroup`, `expand` |
| unsigned on *N* bits | range $[0, 2^N-1]$ | `encode_unsigned` |
| binary arithmetic | column / shift-and-add / long division | `bin_add`, `bin_sub`, `bin_mul`, `bin_div` |

**Next:** `02_signed_integers_and_ieee754.ipynb` – negative numbers (two's complement) and floating point.